In [1]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcraft.trainer import ThermodynamicTrainer
from imolcraft.trainer.dmff_utils import parser_dmffyaml
from imolcraft.trainer.loss import loss_thermodynamicperturbation
from functools import partial

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 0.85,
  'nonbondedmethod': 'LJPME',
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 50,
  'dispcorr': True,
  'anneal_totaltime': 0},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 24.188},
  'Lc_A': {'weight': 1.0, 'gt': 20.108},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [3]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [4]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charge", "NonbondedForce/epsilon", "NonbondedForce/sigma"],
    label="2rdf_1adf_density",
)

In [5]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1097, 0.1467, 0.1153, 0.1538, 0.1719, 0.1453, 0.1612], dtype=float64),
  'k': Array([314553.12, 247609.12, 746007.2 , 194556.  , 237065.44, 571534.4 ,
         335473.12], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([1.87762521, 1.90956473, 1.91637152, 1.94970731, 3.11680898,
         2.08881005, 1.86785137, 1.69183491, 2.09526777, 1.85947379],      dtype=float64),
  'k': Array([ 326.352   ,  409.1952  ,  391.6224  ,  554.7984  ,  612.5376  ,
          530.5312  , 1026.7536  ,  567.551232, 1072.7776  , 1089.5136  ],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([0.        , 0.        , 0.        , 3.14159265, 3.14159265,
         4.68141978, 4.82988662], dtype=float64),
  'proper_k': Array([ 0.50208   ,  0.65084444,  0.65084444,  0.        ,  0.        ,
         27.18068704, 37.90498239], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dtype=flo

In [6]:
from imolcraft.trainer.dmff_utils import neutralize

def grad_modify(grads):
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[10].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[11].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[12].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[13].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[14].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[15].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[16].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[17].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[18].set(0.0)
    # grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[19].set(0.0)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list)

    return ffparams

In [7]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [8]:
trainer.setup()

Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Potential Energy (kJ/mole)","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,-27688.685604905637,1.500084829780657,0,--
2000,-27787.578976487668,1.494784278331535,288,0:35
3000,-27759.923428391965,1.502578973879205,281,0:35
4000,-27737.77785954431,1.5009629817488461,276,0:36
5000,-27712.542218064817,1.487595087612511,276,0:36
6000,-27670.650311326535,1.490787325612197,276,0:35
7000,-27688.158505296262,1.4836342272104552,277,0:35
8000,-27764.743038987668,1.4935775112585201,277,0:34
9000,-27715.726063584836,1.5032228746227907,276,0:34
10000,-27812.98292241052,1.4967087752310921,276,0:34
11000,-27772.334378098996,1.4919932897138133,277,0:33
12000,-27652.09682927087,1.4884787004772666,277,0:33
13000,-27658.557736253293,1.4824678075540112,276,0:33
14000,-27719.405575608762,1.490995434396124,276,0:33
15000,-27772.56985173181,1.488594166433345,276,0:32
16000,-27684.77843937829

/Users/srak/miniconda3/envs/imc/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:261: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 100/100 [00:00<00:00, 209.83it/s]


In [9]:
trainer.fit(10, 2)

Pair list: 100%|██████████| 100/100 [00:17<00:00,  5.59it/s]


Init Energy calc: Memory Usage: 2687.48 MB
Energy calc: 0.70 sec. Memory Usage: 3327.83 MB
Get weight, Memory Usage: 4487.84 MB
Finish loss calc Memory Usage: 4517.77 MB
grad obtained.... Memory Usage: 4739.73 MB
grad modify.... Memory Usage: 4841.58 MB
update finished.... Memory Usage: 4873.70 MB
Effective sample sizes:
  sample_0: 98
  Total: 99.99080214332992
Epoch 0, Loss: 1.0780544709937607, Memory Usage: 4879.67 MB
Loss:  1.0780544709937607
Best Loss:  1.0780544709937607 at epoch  0
Epoch 0 completed in 26.89 seconds.
----
Init Energy calc: Memory Usage: 4879.67 MB
Energy calc: 0.00 sec. Memory Usage: 4896.59 MB
Get weight, Memory Usage: 4880.80 MB
Finish loss calc Memory Usage: 4881.00 MB
grad obtained.... Memory Usage: 4881.41 MB
grad modify.... Memory Usage: 4881.41 MB
update finished.... Memory Usage: 4881.41 MB
Effective sample sizes:
  sample_0: 96
  Total: 97.07152862576544
Loss:  1.0621088527280196
Best Loss:  1.0621088527280196 at epoch  1
Epoch 1 completed in 2.83 secon

100%|██████████| 100/100 [00:00<00:00, 185.92it/s]


Loss:  1.0034020561710821
Best Loss:  1.0034020561710821 at epoch  5
Epoch 5 completed in 43.12 seconds.
----


Pair list: 100%|██████████| 100/100 [00:17<00:00,  5.82it/s]


Init Energy calc: Memory Usage: 6023.34 MB
Energy calc: 0.47 sec. Memory Usage: 6088.17 MB
Get weight, Memory Usage: 7016.52 MB
Finish loss calc Memory Usage: 7016.52 MB
grad obtained.... Memory Usage: 7035.42 MB
grad modify.... Memory Usage: 7193.50 MB
update finished.... Memory Usage: 7193.50 MB
Effective sample sizes:
  sample_0: 98
  Total: 99.16027072880794
Epoch 6, Loss: 1.0367265942310384, Memory Usage: 7193.50 MB
Loss:  1.0367265942310384
Best Loss:  1.0034020561710821 at epoch  5
Epoch 6 completed in 21.23 seconds.
----
Init Energy calc: Memory Usage: 7193.50 MB
Energy calc: 0.00 sec. Memory Usage: 7207.52 MB
Get weight, Memory Usage: 7193.50 MB
Finish loss calc Memory Usage: 7193.50 MB
grad obtained.... Memory Usage: 7193.50 MB
grad modify.... Memory Usage: 7193.50 MB
update finished.... Memory Usage: 7193.50 MB
Effective sample sizes:
  sample_0: 96
  Total: 97.5016470027123
Loss:  1.0301930710528358
Best Loss:  1.0034020561710821 at epoch  5
Epoch 7 completed in 2.79 second

In [10]:
trainer.get_loss_gradients()

Init Energy calc: Memory Usage: 7881.05 MB
Energy calc: 0.00 sec. Memory Usage: 7951.39 MB
Get weight, Memory Usage: 7881.05 MB
Finish loss calc Memory Usage: 7881.05 MB


(Array(1.00888356, dtype=float64),
 {'HarmonicAngleForce': {'angle': Array([ 1.31652769, -1.98754128,  1.44884246, -1.42745495,  4.21890567,
          -0.76855451,  1.55848175, -1.56118635, -0.47393294,  1.03013329],      dtype=float64),
   'k': Array([ 1.17086665e-04, -3.65720066e-04, -4.48715290e-04,  2.51273090e-04,
           4.67647097e-04, -8.74614241e-05, -1.60863307e-04, -7.04706181e-06,
          -7.72905743e-05,  4.76323397e-05], dtype=float64)},
  'HarmonicBondForce': {'k': Array([-1.49081525e-08, -1.65662353e-07,  1.68245145e-07,  2.95764807e-07,
           3.42674737e-07,  2.44190798e-07,  2.53125148e-07], dtype=float64),
   'length': Array([-10.78449365, -71.69920811, -48.59956124, -42.1182084 ,
           22.72363051,   3.37894849,  52.28822596], dtype=float64)},
  'NonbondedForce': {'charge': Array([ 0.48738684,  0.50302945,  0.53426344,  0.0480248 ,  0.24038531,
           0.07824862,  0.36400728,  0.59238218,  1.4149442 ,  0.66710036,
           0.14695143, -0.3647353

In [11]:
trainer.fit(490, 2)

Init Energy calc: Memory Usage: 7881.05 MB
Energy calc: 0.00 sec. Memory Usage: 7954.48 MB
Get weight, Memory Usage: 8246.12 MB
Finish loss calc Memory Usage: 8246.12 MB
grad obtained.... Memory Usage: 8246.12 MB
grad modify.... Memory Usage: 8246.12 MB
update finished.... Memory Usage: 8246.12 MB
Effective sample sizes:
  sample_0: 70
  Total: 70.85644506586968
Loss:  1.0088835575560775
Best Loss:  1.0034020561710821 at epoch  5
Epoch 11 completed in 2.79 seconds.
----
Init Energy calc: Memory Usage: 8246.12 MB
Energy calc: 0.00 sec. Memory Usage: 8317.47 MB
Get weight, Memory Usage: 8246.12 MB
Finish loss calc Memory Usage: 8246.12 MB
grad obtained.... Memory Usage: 8246.12 MB
grad modify.... Memory Usage: 8246.12 MB
update finished.... Memory Usage: 8246.12 MB
Effective sample sizes:
  sample_0: 63
  Total: 63.28616618558411
Epoch 12, Loss: 1.0052616417346334, Memory Usage: 8246.12 MB
Loss:  1.0052616417346334
Best Loss:  1.0034020561710821 at epoch  5
Epoch 12 completed in 2.91 sec

100%|██████████| 100/100 [00:00<00:00, 171.62it/s]


Loss:  0.9983011698362594
Best Loss:  0.9983011698362594 at epoch  15
Epoch 15 completed in 43.01 seconds.
----


Pair list: 100%|██████████| 100/100 [00:16<00:00,  6.01it/s]


Init Energy calc: Memory Usage: 8698.89 MB
Energy calc: 0.50 sec. Memory Usage: 8792.61 MB
Get weight, Memory Usage: 8748.97 MB
Finish loss calc Memory Usage: 8748.97 MB
grad obtained.... Memory Usage: 8751.50 MB
grad modify.... Memory Usage: 8751.50 MB
update finished.... Memory Usage: 8751.50 MB
Effective sample sizes:
  sample_0: 96
  Total: 97.88330725703045
Epoch 16, Loss: 0.9830143590402107, Memory Usage: 8751.50 MB
Loss:  0.9830143590402107
Best Loss:  0.9830143590402107 at epoch  16
Epoch 16 completed in 20.70 seconds.
----
Init Energy calc: Memory Usage: 8751.50 MB
Energy calc: 0.00 sec. Memory Usage: 8769.64 MB
Get weight, Memory Usage: 8751.50 MB
Finish loss calc Memory Usage: 8751.50 MB
grad obtained.... Memory Usage: 8751.50 MB
grad modify.... Memory Usage: 8751.50 MB
update finished.... Memory Usage: 8751.50 MB
Effective sample sizes:
  sample_0: 96
  Total: 97.49399174088867
Loss:  0.9752497643440678
Best Loss:  0.9752497643440678 at epoch  17
Epoch 17 completed in 2.80 

...

Init Energy calc: Memory Usage: 10289.11 MB
Energy calc: 0.49 sec. Memory Usage: 10340.27 MB
Get weight, Memory Usage: 10257.47 MB
Finish loss calc Memory Usage: 10257.47 MB
grad obtained.... Memory Usage: 10260.47 MB
grad modify.... Memory Usage: 10260.47 MB
update finished.... Memory Usage: 10260.47 MB
Effective sample sizes:
  sample_0: 81
  Total: 82.80355306362254
Loss:  0.2124335964732946
Best Loss:  0.19678767328257685 at epoch  416
Epoch 417 completed in 18.19 seconds.
----
Init Energy calc: Memory Usage: 10260.47 MB
Energy calc: 0.00 sec. Memory Usage: 10285.12 MB
Get weight, Memory Usage: 10215.78 MB
Finish loss calc Memory Usage: 10215.78 MB
grad obtained.... Memory Usage: 10215.78 MB
grad modify.... Memory Usage: 10215.78 MB
update finished.... Memory Usage: 10215.78 MB
Effective sample sizes:
  sample_0: 82
  Total: 83.56151274916319
Epoch 418, Loss: 0.2113781772283398, Memory Usage: 10215.78 MB
Loss:  0.2113781772283398
Best Loss:  0.19678767328257685 at epoch  416
Epoch 

100%|██████████| 100/100 [00:00<00:00, 249.99it/s]


Loss:  0.19595154013388147
Best Loss:  0.19595154013388147 at epoch  469
Epoch 469 completed in 44.16 seconds.
----


Pair list: 100%|██████████| 100/100 [00:13<00:00,  7.18it/s]


Init Energy calc: Memory Usage: 10223.02 MB
Energy calc: 0.49 sec. Memory Usage: 10270.33 MB
Get weight, Memory Usage: 10899.86 MB
Finish loss calc Memory Usage: 10899.86 MB
grad obtained.... Memory Usage: 10902.42 MB
grad modify.... Memory Usage: 10902.42 MB
update finished.... Memory Usage: 10902.42 MB
Effective sample sizes:
  sample_0: 84
  Total: 85.10815053139177
Epoch 470, Loss: 0.19805561425515833, Memory Usage: 10680.11 MB
Loss:  0.19805561425515833
Best Loss:  0.19595154013388147 at epoch  469
Epoch 470 completed in 18.99 seconds.
----
Init Energy calc: Memory Usage: 10680.11 MB
Energy calc: 0.00 sec. Memory Usage: 10695.61 MB
Get weight, Memory Usage: 10680.11 MB
Finish loss calc Memory Usage: 10680.11 MB
grad obtained.... Memory Usage: 10680.11 MB
grad modify.... Memory Usage: 10680.11 MB
update finished.... Memory Usage: 10680.11 MB
Effective sample sizes:
  sample_0: 84
  Total: 85.60144569059432
Loss:  0.19758463129004172
Best Loss:  0.19595154013388147 at epoch  469
Epo